In [1]:
import requests
from bs4 import BeautifulSoup
import json
import csv
import os
from urllib.parse import urlparse
from pathlib import Path

# Set up headers for the request
headers = {'Accept': '*/*', 'Connection': 'keep-alive', 'User-Agent': 
    'Mozilla/5.0 (Windows NT 6.0) AppleWebKit/537.36 (KHTML,like Gecko) Chrome/70.0.3538.110 Safari/537.36', 
    'Accept-Language':'en-US;q=0.5,en;q=0.3', 'Cache-Control': 'max-age=0', 'Upgrade-Insecure-Requests': '1'}




In [2]:
def download_image(image_url, show_name, folder_path):
    """
    Download image from URL and save it with show name as filename
    """
    try:
        # Create the images directory if it doesn't exist
        Path(folder_path).mkdir(parents=True, exist_ok=True)
        
        # Clean the show name to create a valid filename
        valid_filename = "".join(x for x in show_name if x.isalnum() or x in (' ', '-', '_')).rstrip()
        
        # Get file extension from URL
        ext = os.path.splitext(urlparse(image_url).path)[1]
        if not ext:
            ext = '.jpg'  # Default to jpg if no extension found
            
        # Create full file path
        file_path = os.path.join(folder_path, f"{valid_filename}{ext}")
        
        # Download and save the image
        response = requests.get(image_url, headers=headers)
        response.raise_for_status()
        
        with open(file_path, 'wb') as f:
            f.write(response.content)
            
        return file_path
    except Exception as e:
        print(f"Error downloading image for {show_name}: {str(e)}")
        return None

In [3]:
# Get the response from the IMDb Top TV Chart page
response = requests.get('https://www.imdb.com/chart/toptv/', headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Find the script tag containing the JSON data
script = soup.find('script', {'type': "application/ld+json"})

# Check if the script tag is found and contains data
if script:
    data = json.loads(script.text)   
    
    # Check if 'itemListElement' exists
    if 'itemListElement' in data:
        print(json.dumps(data, indent=4))
        # Define the folder for saving images
        image_folder = 'imdb_show_images'
        # Open the CSV file in write mode
        with open('top_tv_shows_full_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
            # Create a CSV writer
            csvwriter = csv.writer(csvfile)
            
            # Write the header row (fields)
            csvwriter.writerow(['Rank', 'Show Name', 'URL', 'Description', 'Genre', 'Rating', 'Votes', 'Content Rating', 'Image URL'])
            
            # Write the data for each TV show
            for idx, item in enumerate(data['itemListElement'][:250], start=1):
                show_item = item.get('item', {})
                show_name = show_item.get('name', 'N/A')
                show_url = show_item.get('url', '')
                show_description = show_item.get('description', 'N/A')
                show_genre = show_item.get('genre', 'N/A')
                show_rating = show_item.get('aggregateRating', {}).get('ratingValue', 'N/A')
                show_votes = show_item.get('aggregateRating', {}).get('ratingCount', 'N/A')
                show_content_rating = show_item.get('contentRating', 'N/A')
                show_image_url = show_item.get('image', 'N/A')
                
                # Download the image and get local path
                local_image_path = 'N/A'
                if show_image_url != 'N/A':
                    local_image_path = download_image(show_image_url, show_name, image_folder)

                # Write the row with the rank, show name, URL, description, genre, rating, votes, content rating, and image URL
                csvwriter.writerow([idx, show_name, show_url, show_description, show_genre, show_rating, show_votes, show_content_rating, show_image_url])
        
        print("Data successfully written to 'top_tv_shows_full_data.csv'.")
    else:
        print("No 'itemListElement' found in the JSON data.")
else:
    print("Script containing JSON data not found.")

{
    "@type": "ItemList",
    "itemListElement": [
        {
            "@type": "ListItem",
            "item": {
                "@type": "TVSeries",
                "url": "https://www.imdb.com/title/tt0903747/",
                "name": "Breaking Bad",
                "description": "A chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling methamphetamine with a former student to secure his family&apos;s future.",
                "image": "https://m.media-amazon.com/images/M/MV5BMzU5ZGYzNmQtMTdhYy00OGRiLTg0NmQtYjVjNzliZTg1ZGE4XkEyXkFqcGc@._V1_.jpg",
                "aggregateRating": {
                    "@type": "AggregateRating",
                    "bestRating": 10,
                    "worstRating": 1,
                    "ratingValue": 9.5,
                    "ratingCount": 2276241
                },
                "contentRating": "TV-MA",
                "genre": "Crime, Drama, Thriller"
            }
        },
        {
            "